# Convert the CSV Data to JSON

This notebook is used to convert the CSV data into a nested JSON. The basic steps are as follows:

- Read the CSV files into Pandas DataFrames.
- Handle calendar dates.
- Create a nested dictionary structure that contains all the information from the different CSV files.
- Save the nested dictionary structure as a JSON file.

## Understanding the RAW CSV files

### party.csv

- This is the main file.
- Each line represents a *vl:Identity* of a party.
- In the field `party_name_id` are some versions of the same party with different names.

## version.link

The RDF data of the political parties is modeled using the [version.link schema](https://version.link). The main idea of this schema is to have both a representation of the current state of the party, but also a representation of the historical changes of the party.

Each party exists as *one* vl:Identity with a unique ID. The vl:Identity represents the current state of the party. The historical changes of the party are represented as vl:Version instances. The following changes trigger the creation of a new vl:Version instance:

- the party changes its **name**
- the party starts or stops being a **child of** another party
- the party starts or stops being an **observer in** another party
- the party starts or stops being **affiliated with** another party
- the party is involved in a **successor-of**, **split-from** or **accessed** event

The vl:Version instances also have a unique ID built from the party's ID and the starting date of the version.

### Changes to the "pure" version.link schema

The original idea of the version.link schema is, that all the information on the vl:Identiy is coming from a vl:Version. This is not the case in the political party data. The vl:Identity contains information that is not available in a versioned form. An example would be the some alternate identifiers.

### Relationships and Events

The original data in the `relationship.csv` file contains data that are from a version.link perspective two different types of relationships: Such events that have a start and stop date (e.g. "child of", "observer in", "affiliated with") and events that are instantaneously one-time events (e.g. "successor of", "split from", "accessed").

The solution chosen here is to create events from the relationships so that e.g a "child of" relationship results in a child-of-start-event and a child-of-stop-event. On the other hand, a "successor of" relationship results only in one event. Each vl:Version is started and stopped by events (can be multiple events for starting and stopping a vl:Version). Furthermore, name changes are also represented as events (name-change-start-event and name-change-stop-event).

### Hierarchies

Versioned data in a hierarchical structure is potentially complex. If a new relationship (e.g. child-of) triggers also on the parent side a new version, this changed parent version will consequently also trigger new versions on *all* children. This leads to a not very useful **cascade of new versions**. Therefore, the approach chosen here is to create only a new version on the lower hierarchy level (e.g. on the child level). The link to the parent will be done to the vl:Identity and not to the vl:Version of the parent.

### Alternative Rejected Approach

An alternative approach would be to have only one version per identity. The version has got all the history, the identity only the currently valid data. This would resemble more of a "Wikidata" approach.This aproach was tried in `1_convert_b.ipynb` (see Git history).

The problem was, that this approach made it difficult to build vl:successor and vl:predecessor relationships, because the vl:successor can be the same party (if a split happens, the new party is one follower, but the existing party is also a follower, because it is still existing). This was difficult to model with only one version per identity.

## Create Dictionaries

In the following part, all information from the different CSV files is combined into a **single nested dictionary** structure. This is done by iterating over the rows of the `party_csv` (which is the "main" file) and fetching all the related information from the other connected CSV files.

## Download CSV Files as ZIP and Extract them

In [58]:
from dotenv import load_dotenv
import os

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

True

In [63]:
import requests
from pathlib import Path

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

url = os.getenv('AP_DOWNLOAD_URL')
auth = (os.getenv('AP_USER'), os.getenv('AP_PASSWORD'))

with requests.get(url, auth=auth, stream=True) as r:
    r.raise_for_status()
    with open(CSV_PATH / "data.zip", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [64]:
import zipfile

with zipfile.ZipFile(CSV_PATH / "data.zip", "r") as zf:
    zf.extractall(CSV_PATH)

### Read CSV Files

In [65]:
import pandas as pd

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

# main file
party_csv = pd.read_csv(CSV_PATH / "party.csv")

# names
party_name_csv = pd.read_csv(CSV_PATH / "party_name.csv")

# relationships
relationship_csv = pd.read_csv(CSV_PATH / "relationship.csv")
relationship_type_csv = pd.read_csv(CSV_PATH / "relationship_type.csv")

# party identifiers
party_code_csv = pd.read_csv(CSV_PATH / "party_code.csv")
party_code_type_csv = pd.read_csv(CSV_PATH / "party_code_type.csv")

# areas
party_area_csv = pd.read_csv(CSV_PATH / "party_area.csv")
area_csv = pd.read_csv(CSV_PATH / "area.csv")
area_type_csv = pd.read_csv(CSV_PATH / "area_type.csv")

### Get Overview of the CSV Files

In [66]:
from IPython.display import display

display(party_csv)

,party_id,identifier,color,show_in_cantonal_table_and_graphs,party_name_id,relationship_id_1,relationship_id_2,social_group_id,party_category_id,election_result_id,...,founding_date_datetime,founding_date_notes,founding_date_sources,dissolution_date_precision,dissolution_date_datetime,dissolution_date_notes,dissolution_date_sources,translations_languages_code3,translations_description3,translations_hls_link_external_url3
0,1,svp-ch,#4B8A3E,NaN,[1],"[3,9]","[5,12,35,69,90,98,141,174,192,193,232,260,276,...",[1],[16],"[1,2,3,4,5,6,7,8,9,10,11,12,13,14,106,107,108,...",...,1971-09-22T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,sp-ch,#F0554D,NaN,[2],"[618,635]","[13,36,70,91,99,142,175,196,201,202,233,234,26...",[2],[1],"[15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,...",...,1888-10-21T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,fdplib-ch,#3872B5,NaN,[3],"[1,6]","[18,38,72,93,101,135,144,177,203,205,210,240,2...",[3],[15],"[43,44,45,46,148,149,150,151,6290,7226,7227,72...",...,2009-01-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,mitte-ch,#D6862B,NaN,[4],"[2,4]","[21,31,40,43,61,75,95,103,125,145,146,163,180,...",[4],[14],"[47,152,6284,7250,7251,7252,7253,10247,15123,1...",...,2021-01-01T12:00:00,NaN,page_number_string: 360\r\nnotes: \r\nsource_f...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,gruene-ch,#84B547,NaN,"[6,7,8,9,199]",[],"[22,44,45,48,49,53,54,56,78,79,96,104,105,106,...",[5],[13],"[48,49,50,51,52,53,54,55,56,57,58,59,60,153,15...",...,1983-05-28T12:00:00,NaN,page_number_string: 3\r\nnotes: \r\nsource_ful...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432,445,edu-sz,#A65E42,1.0,[712],[687],[],[441],[438],[],...,2026-09-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
433,446,fvp-bl,#4785C2,1.0,"[713,714]",[691],[689],[442],[439],[],...,1905-07-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,1919-08-03T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN
434,447,dvv-bl,#295FA8,1.0,"[715,716]",[],[690],[443],[440],[],...,1910-07-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,1919-08-03T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN
435,448,auto-so,NaN,NaN,"[719,720]",[693],[],[444],[441],"[15130,15131,15132,15133,15134,15135,15136,151...",...,1987-01-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,unknown,2000-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN


In [67]:
display(relationship_csv)

,relationship_id,party_1_id,party_2_id,type,notes,sources,from_date_precision,from_date_datetime,from_date_notes,from_date_sources,to_date_precision,to_date_datetime,to_date_notes,to_date_sources
0,1,3,9,2,NaN,NaN,exact,2009-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,2,4,10,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,2021-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,11,2,NaN,page_number_string: 98\r\nnotes: \r\nsource_fu...,exact,1971-09-22T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4,13,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,2021-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
4,5,13,1,3,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,2008-11-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,689,105,446,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,1919-08-03T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
680,690,105,447,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,exact,1919-08-03T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
681,691,446,9,1,NaN,page_number_string: \r\nnotes: \r\nsource_full...,multiyear,1906-01-01T12:00:00,NaN,NaN,exact,1919-08-03T12:00:00,NaN,NaN
682,692,303,302,3,NaN,page_number_string: \r\nnotes: \r\nsource_full...,year,2006-07-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN


### Date Handling

All the dates in the CSV and after import to a pandas DataFrame are given as strings in the form of "YYYY-MM-DDT12:00:00" format (always with the time set to 12:00:00). The time part is not relevant for the use case, so it will be removed. In the final RDF data, the end dates will be shifted by one day as the `schema:validThrough` property is used, which denotes the last valid day.

- `schema:validFrom` means the first valid day
- `schema:validThrough` means the last valid day

Unknown dates are represented as `NaN` in the DataFrame. To check for unknown dates, the `pd.isna()` function is used instead of checking for `None`, as the latter does not work with `NaN` values.

In [68]:
def convert_time_column(df, column_name):
    """
    Convert the time column to a date format and remove the time part. 
    Convert it back to strings in the format YYYY-MM-DD.
    """
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce').dt.date
    df[column_name] = df[column_name].astype(str)
    return df

In [69]:
# convert the time columns to date format and remove the time part

party_csv = convert_time_column(party_csv, 'founding_date_datetime')
party_csv = convert_time_column(party_csv, 'dissolution_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'from_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'to_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'from_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'to_date_datetime')

### Create Parties Dictionary

Helper functions:

In [70]:
def parse_list(s):
    """
    Parse a string representation of a list like "[1, 2, 3]" into a Python list.
    """
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [71]:
def is_currently_valid(day, valid_from, valid_through):
    """
    Check if a given day is within the valid range defined by valid_from and valid_through.
    valid_through can be NaN to indicate that the validity is open-ended. 
    """
    return valid_from <= str(day) and (pd.isna(valid_through) or str(day) < valid_through)

In [72]:
def map_area_id_to_uri(area_type, area_id):
    """
    Map an area_id to its corresponding URI.
    """
    if area_type == "country" and area_id == 1: # Switzerland
        return "https://ld.admin.ch/country/CHE"
    elif area_type == "canton": # Cantons
        canton_id = area_id - 1
        return f"https://ld.admin.ch/canton/{canton_id}"
    else:
        return None

In [73]:
# create the main dictionary that will hold all the parties and their information
parties = {}

# for each identity (each row in the party_csv)
for _, row in party_csv.iterrows():
    
    # create a dictionary for the current party
    party = {
        "identity": {},
        "versions": [],
        "debug": {}
    }
    
    # get the party_id - needed to set the key in the parties dictionary
    party_id = row["party_id"]

    # add basic date information
    party["identity"]["founding_date"] = row["founding_date_datetime"]
    party["identity"]["dissolution_date"] = row["dissolution_date_datetime"] #if not available, it will be NaN

    
    #### names ####
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    # all names are stored in a list of dictionaries
    names = []

    for party_name_id in party_name_ids:
        
        # names
        name_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_name{lang_id}"].item()
            if str(lang_name) != "nan" and str(name) != "nan":
                name_lang[str(lang_name)[0:2]] = name
        
        # abbreviations
        abbr_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_abbreviation{lang_id}"].item()
            if str(lang_abbr) != "nan" and str(abbr) != "nan":
                abbr_lang[str(lang_abbr)[0:2]] = abbr
        
        # dates for the name and abbreviation
        valid_from = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        valid_through = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        
        # create a name entry and add it to the names list
        name_entry = {"id": party_name_id, "name": name_lang, "abbr": abbr_lang, "valid_from": valid_from, "valid_through": valid_through}
        names.append(name_entry)

    
    # sort names by from_date
    names.sort(key=lambda x: x["valid_from"])
    
    # add names to the party dictionary
    party["debug"]["names"] = names

    # add last name and abbreviation to the party dictionary (the currently valid ones)
    current_name = names[-1] if names else {}
    party["identity"]["name"] = current_name.get("name", {})
    party["identity"]["abbr"] = current_name.get("abbr", {})
    
    #### relationships ####

    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2 depending on who is the "party from".
    relationships = []

    # get relationships from both columns and combine them into a single list
    relationship_ids = parse_list(row["relationship_id_1"]) + parse_list(row["relationship_id_2"])

    for relationship_id in relationship_ids:
        
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        valid_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        valid_through = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        
        relationship_entry = {"id": relationship_id, 
                              "type_label": relationship_label,
                              "party_from": relationship_from, 
                              "party_to": relationship_to, 
                              "valid_from": valid_from, 
                              "valid_through": valid_through}
        
        if relationship_entry["type_label"] in ["successor-of", "split-from", "accessed"]:
            relationships.append(relationship_entry)

        if relationship_entry["type_label"] in ["child-of", "observer-in", "affiliated-with"]:

            # hierarchical relationships are only added on one side because of "version-avalanche" effect.
            if relationship_entry["party_to"] != party_id:
                relationships.append(relationship_entry)

    # sort relationships by from_date and add them to the party dictionary
    relationships.sort(key=lambda x: x["valid_from"])
    
    # add relationships to the party dictionary
    party["debug"]["relationships"] = relationships

    
    #### party identifiers ####

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    party["identity"]["party_codes"] = party_codes


    #### party color ####
    
    party["identity"]["color"] = row["color"]  # add the color from the party.csv row


    #### areas ####

    party_area_id = parse_list(row["party_area_id"])
    area_id = party_area_csv.loc[party_area_csv["id"] == party_area_id[0], "area_area_id"].item()
    area_type_id = area_csv.loc[area_csv["area_id"] == area_id, "area_type_id"].item()
    area_type = area_type_csv.loc[area_type_csv["area_type_id"] == area_type_id, "identifier"].item()
    area_uri = map_area_id_to_uri(area_type, area_id)
    party["identity"]["area"] = {
        "type": area_type,
        "uri": area_uri
    }


    #### events ####

    events = []

    # add name change events
    for name in names:

        # name start event
        events.append({
            "event_type": "name-change_start",
            "corresponding_id": name["id"],
            "date": name["valid_from"],
        })
        # name stop event if valid_through is not NaN
        if not pd.isna(name["valid_through"]):
            events.append({
                "event_type": "name-change_stop",
                "corresponding_id": name["id"],
                "date": name["valid_through"],
            })

    # add relationship events
    for relationship in relationships:

        # if the relationship is shape-changing, this is an instantaneous event, so we only add one event for it.
        if relationship["type_label"] in ["successor-of", "split-from", "accessed"]:
            events.append({
                "event_type": relationship["type_label"],
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })

        # hierarchical relationships are ongoing events, so we add a start and stop event (if applicable) for them.
        if relationship["type_label"] in ["child-of", "observer-in", "affiliated-with"]:
            events.append({
                "event_type": f"{relationship["type_label"]}_start",
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })
            if not pd.isna(relationship["valid_through"]):
                events.append({
                    "event_type": f"{relationship["type_label"]}_stop",
                    "corresponding_id": relationship["id"],
                    "date": relationship["valid_through"],
                    "party_from": relationship["party_from"],
                    "party_to": relationship["party_to"]
                })

    # add dissolution event if it exists 
    # (it can be followed by a successor event even years later, 
    # so it is not necessarily the last event)
    if not pd.isna(party["identity"]["dissolution_date"]):
        events.append({
            "event_type": "dissolution",
            "date": party["identity"]["dissolution_date"],
        })

    # sort events by date
    events.sort(key=lambda x: x["date"])

    # add founding event at position 0 (first event)
    events.insert(0, {
        "event_type": "founding",
        "date": party["identity"]["founding_date"],
    })

    # events will only be added to the party dictionary after the mapping 
    # to started and stopped versions is done, because the version_ids are 
    # needed for that mapping. 


    #### event_dates and version_ids #### 

    # list all unique event dates in sorted order, this will be used to create version_ids
    event_dates = sorted(set([event["date"] for event in events]))
    party["debug"]["event_dates"] = event_dates

    version_ids = event_dates.copy()

    # if there is a dissolution event, there are no versions with this date 
    # or even later (late succession events are not considered to 
    # create new versions, because the party is already dissolved).
    if not pd.isna(party["identity"]["dissolution_date"]):
        version_ids = [date for date in version_ids if date < party["identity"]["dissolution_date"]]

    party["debug"]["version_ids"] = version_ids


    #### map events to started or stopped versions ####

    for event in events:

        # map the event to the version that started or stopped with this event.
        
        if event["date"] == version_ids[0]:  # if the event date is the same date as the first version, there is no version before this event
            version_before_event = None  # there is no version before the first event
        else:
            event_index = event_dates.index(event["date"])
            try:
                version_before_event = version_ids[event_index - 1]  # the version that was valid before this event
            except IndexError:
                version_before_event = version_ids[-1]  # if the event is after the last version, the last version is the one that was valid before this event

        try:
            version_after_event = version_ids[event_dates.index(event["date"])]  # the version that will be valid after this event
        except IndexError:
            version_after_event = None

        event["started_version"] = version_after_event
        event["stopped_version"] = version_before_event


        if event["event_type"] == "successor-of" and event["party_from"] != party_id:
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "split-from" and event["party_from"] != party_id:
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "accessed":
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event
            
    # add events to the party dictionary
    party["debug"]["events"] = events


    #### versions ####

    versions = {}

    for i, version_id in enumerate(version_ids):
        versions[version_id] = {
            "valid_from": version_id,
            "valid_through": None,
            "is_founding": False,
            "is_dissolution": False,
            "name": None,
            "abbr": None,
            "relationships": [],
            "successors": []
        }

        # if there is a next version, set the valid_through to the next version's date
        if i < len(version_ids) - 1:
            versions[version_id]["valid_through"] = version_ids[i + 1]

            # regular successor within the same party
            versions[version_id]["successors"].append(f"{party_id}_{version_ids[i + 1]}")

        # if it is the last version, one has to check if there is another event 
        # which would terminate the version, e.g. a dissolution event or a name-change_stop event. 
        # If there is no such event, the valid_through is None, meaning the version is still 
        # ongoing.
        else:
            if len(event_dates) > len(version_ids):
                versions[version_id]["valid_through"] = event_dates[-1]

        # check for outside successors (to other parties) in the events
        for event in events:
            if event["stopped_version"] == version_id and "outside_successor" in event:
                versions[version_id]["successors"].append(event["outside_successor"])

        # add the currently valid name
        for name in names:
            if is_currently_valid(version_id, name["valid_from"], name["valid_through"]):
                versions[version_id]["name"] = name["name"]
                versions[version_id]["abbr"] = name["abbr"]
                break  # only one name should be valid at a time

        # add the currently valid hierarchical relationships
        versions[version_id]["relationships"] = []
        hierarchical_relationships = [r for r in relationships if r["type_label"] in ["child-of", "observer-in", "affiliated-with"]]

        for relationship in hierarchical_relationships:
            if is_currently_valid(version_id, relationship["valid_from"], relationship["valid_through"]):
                versions[version_id]["relationships"].append({
                    "type_label": relationship["type_label"],
                    "party_to": relationship["party_to"]
                })

        # add the starting and stopping events for this version
        versions[version_id]["starting_events"] = [event for event in events if event["started_version"] == version_id]

        # add a flag to indicate if this version is the founding version (if there is a founding event in the starting events)
        if [event for event in versions[version_id]["starting_events"] if event["event_type"] == "founding"]:
            versions[version_id]["is_founding"] = True

        versions[version_id]["stopping_events"] = [event for event in events if event["stopped_version"] == version_id]

        # add a flag to indicate if this version is the dissolution version (if there is a dissolution event in the stopping events)
        if [event for event in versions[version_id]["stopping_events"] if event["event_type"] == "dissolution"]:
            versions[version_id]["is_dissolution"] = True

    # add party versions to the party dictionary
    party["versions"] = versions

    # add relations from last version to the identity of the party for easier access to the current state of the party
    last_version_id = version_ids[-1]
    party["identity"]["relationships"] = party["versions"][last_version_id]["relationships"]

    # add party to parties dictionary (finish line operations)
    parties[party_id] = party


### Create Events Dictionary

In [74]:
import json

events = {

}

event_counter = 1

for party_id, party in parties.items():
    for version_id, version in party["versions"].items():
        full_version_id = f"{party_id}_{version_id}"

        ### start event
        events[str(event_counter)] = {
            "predecessors": [],
            "successors": [full_version_id]
        }
        event_counter += 1

        ### stop event

        if pd.isna(version["valid_through"]):
            continue
        else:  # if the version is still ongoing, there is no stop event to create
            
            events[str(event_counter)] = {
                    "predecessors": [full_version_id],
                    "successors": version["successors"]
            }
            event_counter += 1

In [75]:
import json
import hashlib
from collections import defaultdict

def merge_events(events: dict) -> dict:
    # --- Union-Find ---
    parent = {eid: eid for eid in events}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # Pfadkompression
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # --- Pro Seite: welches Objekt kommt bei welchen Events vor? ---
    for side in ("predecessors", "successors"):
        owner = {}  # Objekt -> erstes Event, das es auf dieser Seite hat
        for eid, ev in events.items():
            for obj in ev.get(side, []):
                if obj in owner:
                    union(owner[obj], eid)  # Konflikt -> gleiches Event
                else:
                    owner[obj] = eid

    # --- Gruppen bilden ---
    groups = defaultdict(list)
    for eid in events:
        groups[find(eid)].append(eid)

    # --- Zusammenführen ---
    merged = {}
    for members in groups.values():
        members.sort()
        new_id = members[0]  # oder z.B. "+".join(members)
        merged[new_id] = {
            "predecessors": sorted({o for m in members for o in events[m].get("predecessors", [])}),
            "successors": sorted({o for m in members for o in events[m].get("successors", [])}),
            "merged_from": members if len(members) > 1 else [],
        }
        merged_versions = "".join(merged[new_id]['predecessors'] + merged[new_id]['successors'])
        hashed_id = hashlib.blake2b(merged_versions.encode(), digest_size=4).hexdigest()
        merged[new_id]["merged_id"] = hashed_id
    return merged


with open("../data/output/events.json") as f:
    events = json.load(f)

merged_events = merge_events(events)

### Write Dictionaries as JSON

In [76]:
# write the parties dictionary to a JSON file and replace all NaN values with None to make it JSON serializable

import json
import math

def replace_nan_with_none(obj):
    # Wenn es ein Dictionary ist: Rekursion für alle Werte
    if isinstance(obj, dict):
        return {k: replace_nan_with_none(v) for k, v in obj.items()}
    
    # Wenn es eine Liste ist: Rekursion für alle Elemente
    elif isinstance(obj, list):
        return [replace_nan_with_none(x) for x in obj]
    
    # Wenn es ein Float ist und NaN: Ersetze durch 'none'
    elif isinstance(obj, float) and math.isnan(obj):
        return None
    
    # Sonst: Gib das Objekt einfach zurück
    else:
        return obj

parties_cleaned = replace_nan_with_none(parties)

# print the parties dictionary as json
with open("../data/output/parties.json", "w", encoding="utf-8") as f:
    json.dump(parties_cleaned, f, indent=2, ensure_ascii=False)

# print the events dictionary as json
with open("../data/output/events.json", "w", encoding="utf-8") as f:
    json.dump(events, f, indent=2, ensure_ascii=False)

# print the merged events dictionary as json
with open("../data/output/events_merged.json", "w") as f:
    json.dump(merged_events, f, indent=2, ensure_ascii=False)